<a href="https://colab.research.google.com/github/subhajitchatterjee07/Poetry-generator/blob/main/notebooks/en/rag_with_hf_and_milvus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using RAG to query fundamental tasks for various roles in NetSuite ERP

Authored by: Subhajit Chatterjee (subhajit.chatterjee@oracle.com)

## Why even use it in the first place (Need)?
- Isn't the global search not enough? Probably, I do not know. But I found certain third parties which emulate this behaviour.
- Also wouldnt it be great to have a default chatbot to ask generic / role specific questions? I read about 'N/llm' (https://docs.oracle.com/en/cloud/saas/netsuite/ns-online-help/article_1028073928.html) but I thought it would be great for User Experience and onboarding new users, who haven't have memorized all the features.

[Milvus](https://milvus.io/) is a popular open-source vector database that powers AI applications with highly performant and scalable vector similarity search. In this tutorial, we will show you how to build a RAG (Retrieval-Augmented Generation) pipeline with Hugging Face and Milvus.

Qwen2.5-1.5B-Instruct is a lightweight, instruction-tuned language model optimized for following prompts and generating grounded answers in resource-constrained environments.

The RAG system combines a retrieval system with an LLM. The system first retrieves relevant documents from a corpus using Milvus vector database, then uses an LLM hosted in house to generate answers based on the retrieved documents.

The above setup is highly resource efficient and also has good accuracy at explaining results.

## Preparation
### Dependencies and Environment

In [1]:
! pip install --upgrade pymilvus sentence-transformers huggingface-hub langchain_community langchain-text-splitters pypdf tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.3/248.3 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.4 MB/s eta 0:00:00


> If you are using Google Colab, to enable the dependencies, you may need to **restart the runtime** (click on the "Runtime" menu at the top of the screen, and select "Restart session" from the dropdown menu).

In addition, we recommend that you configure your [Hugging Face User Access Token](https://huggingface.co/docs/hub/security-tokens), and set it in your environment variables because we will use a LLM from the Hugging Face Hub. You may get a low limit of requests if you don't set the token environment variable.

In [2]:
import os

os.environ["HF_TOKEN"] = "#You can add your token here#"

### Prepare the data

We use the [AI Act PDF](https://artificialintelligenceact.eu/wp-content/uploads/2021/08/The-AI-Act.pdf), a regulatory framework for AI with different risk levels corresponding to more or less regulation, as the private knowledge in our RAG.

In [3]:
%%bash

if [ ! -f "ERP-Fundamentals-(English)---Student-Guide (1).pdf" ]; then
    wget -q https://artificialintelligenceact.eu/wp-content/uploads/2021/08/The-AI-Act.pdf
fi

We use the [`PyPDFLoader`](https://python.langchain.com/v0.1/docs/modules/data_connection/document_loaders/pdf/) from LangChain to extract the text from the PDF, and then split the text into smaller chunks. By default, we set the chunk size as 1000 and the overlap as 200, which means each chunk will nearly have 1000 characters and the overlap between two chunks will be 200 characters.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("ERP-Fundamentals-(English)---Student-Guide (1).pdf")
docs = loader.load()
print(len(docs))

192


In [5]:
print(docs)

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2022-09-20T07:13:42-04:00', 'author': 'User', 'keywords': 'v1.6', 'moddate': '2022-12-20T19:10:12-05:00', 'title': 'ERP Fundamentals - Student Guide', 'source': 'ERP-Fundamentals-(English)---Student-Guide (1).pdf', 'total_pages': 192, 'page': 0, 'page_label': '1'}, page_content='Course Guide \n \n  \nERP: Fundamentals'), Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2022-09-20T07:13:42-04:00', 'author': 'User', 'keywords': 'v1.6', 'moddate': '2022-12-20T19:10:12-05:00', 'title': 'ERP Fundamentals - Student Guide', 'source': 'ERP-Fundamentals-(English)---Student-Guide (1).pdf', 'total_pages': 192, 'page': 1, 'page_label': '2'}, page_content='This printing: September 2022. \n \nCopyright © 2014, 2022, Oracle and/or its affiliates.  \nThis document contains proprietary in

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)

In [7]:
text_lines = [chunk.page_content for chunk in chunks]

### Prepare the Embedding Model
Define a function to generate text embeddings. We use [BGE embedding model](https://huggingface.co/BAAI/bge-small-en-v1.5) as an example, but you can use any embedding models, such as those found on the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard).

In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

def emb_text(text):
    return embedding_model.encode([text], normalize_embeddings=True).tolist()[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generate a test embedding and print its dimension and first few elements.

In [9]:
test_embedding = emb_text("This is a test")
embedding_dim = len(test_embedding)
print(embedding_dim)
print(test_embedding[:10])

384
[-0.07660680264234543, 0.02531672641634941, 0.012505538761615753, 0.004595162346959114, 0.02577998675405979, 0.038167111575603485, 0.08050814270973206, 0.0030353872571140528, 0.024392176419496536, 0.004880355205386877]


## Load data into Milvus

### Create the Collection

In [10]:
from pymilvus import MilvusClient

milvus_client = MilvusClient(uri="./hf_milvus_demo.db")

collection_name = "rag_collection"

> As for the argument of `MilvusClient`:
> - Setting the `uri` as a local file, e.g.`./hf_milvus_demo.db`, is the most convenient method, as it automatically utilizes [Milvus Lite](https://milvus.io/docs/milvus_lite.md) to store all data in this file.
> - If you have a large amount of data, say more than a million vectors, you can set up a more performant Milvus server on [Docker or Kubernetes](https://milvus.io/docs/quickstart.md). In this setup, please use the server uri, e.g.`http://localhost:19530`, as your `uri`.
> - If you want to use [Zilliz Cloud](https://zilliz.com/cloud), the fully managed cloud service for Milvus, adjust the `uri` and `token`, which correspond to the [Public Endpoint and Api key](https://docs.zilliz.com/docs/on-zilliz-cloud-console#cluster-details) in Zilliz Cloud.


Check if the collection already exists and drop it if it does.

In [11]:
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

Create a new collection with specified parameters.

If we don't specify any field information, Milvus will automatically create a default `id` field for primary key, and a `vector` field to store the vector data. A reserved JSON field is used to store non-schema-defined fields and their values.

In [12]:
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP",  # Inner product distance
    consistency_level="Strong",  # Strong consistency level
)

### Insert data
Iterate through the text lines, create embeddings, and then insert the data into Milvus.

Here is a new field `text`, which is a non-defined field in the collection schema. It will be automatically added to the reserved JSON dynamic field, which can be treated as a normal field at a high level.

In [13]:
from tqdm import tqdm

data = []

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({"id": i, "vector": emb_text(line), "text": line})

insert_res = milvus_client.insert(collection_name=collection_name, data=data)
insert_res["insert_count"]

Creating embeddings: 100%|██████████| 260/260 [00:03<00:00, 85.86it/s]


260

## Build RAG

### Retrieve data for a query

Let's specify a question to ask about the corpus.

In [14]:
question = "How do I set group pricing?"

Search for the question in the collection and retrieve the top 3 semantic matches.

In [15]:
search_res = milvus_client.search(
    collection_name=collection_name,
    data=[
        emb_text(question)
    ],  # Use the `emb_text` function to convert the question to an embedding vector
    limit=3,  # Return top 3 results
    search_params={"metric_type": "IP", "params": {}},  # Inner product distance
    output_fields=["text"],  # Return the text field
)

Let's take a look at the search results of the query


In [16]:
import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

[
    [
        "Item Master: Pricing | 91 \n \n \nERP: Fundamentals \nExercise 03: Create a Pricing Group \nTime: 1 minute \nScenario: We want to utilize pricing groups. Pricing group values are used on customer and item \nrecords to enable the creation of customer-specific pricing for items. \n1 Navigate to Setup > Accounting > Accounting Lists > New. \n2 The Add to Accounting Lists page displays, select Pricing Group. \na. In the Pricing Group field, enter the name of Preferred Customer. \n3 Click Save. \n4 Click the List link, top-right hand side \n5 If necessary, change the Type filter to Pricing Group, to view the list of Pricing Groups and \nconfirm that you see your group. \n6 End.",
        0.8022658228874207
    ],
    [
        "96 | Item Master: Pricing  \n \n \nERP Fundamentals \nExercise 07: Set Up Pricing on a Customer Record \nTime: 2-3 minutes \nScenario: In this exercise, we will associate a pricing on Customer Record, TEST ABC: \n\u26ab Select Price Level \n\u26ab De

### Use LLM to get an RAG response

Before composing the prompt for LLM, let's first flatten the retrieved document list into a plain string.

In [17]:
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

Define prompts for the Language Model. This prompt is assembled with the retrieved documents from Milvus.

In [23]:
PROMPT = """
You are an assistant that answers questions *only* using the provided context.
If the context does not contain the answer, reply: "The information is not available in the provided documents."

<context>
{context}
</context>
<question>
{question}
</question>

"""

In [27]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Smaller alternative to Mixtral (fits in Colab / small GPU with quantization)
#model_id = "EleutherAI/gpt-neo-1.3B"
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# Download + load model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",   # uses GPU if available, else CPU
    torch_dtype="auto"
)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [28]:
prompt = PROMPT.format(context=context, question=question)

In [29]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.3,
    do_sample=True,
    top_p=0.9
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n===== RAG Answer =====\n")
print(answer)


===== RAG Answer =====


You are an assistant that answers questions *only* using the provided context. 
If the context does not contain the answer, reply: "The information is not available in the provided documents."

<context>
Item Master: Pricing | 91 
 
 
ERP: Fundamentals 
Exercise 03: Create a Pricing Group 
Time: 1 minute 
Scenario: We want to utilize pricing groups. Pricing group values are used on customer and item 
records to enable the creation of customer-specific pricing for items. 
1 Navigate to Setup > Accounting > Accounting Lists > New. 
2 The Add to Accounting Lists page displays, select Pricing Group. 
a. In the Pricing Group field, enter the name of Preferred Customer. 
3 Click Save. 
4 Click the List link, top-right hand side 
5 If necessary, change the Type filter to Pricing Group, to view the list of Pricing Groups and 
confirm that you see your group. 
6 End.
96 | Item Master: Pricing  
 
 
ERP Fundamentals 
Exercise 07: Set Up Pricing on a Customer Record 
Tim